файл для работы только “SharafutdinovNil_Task4.ipynb”
Задание: Проверяем данные с помощью Great Expectations
Используйте тот же dataset1.csv. Вам нужно создать Expectations Suite по требованиям к музыкальному датасету, сгенерировать HTML-отчёт (Data Docs) и JSON-файл с результатами.
Выполните шаги:

Создайте Expectations Suite — набор ожидаемых проверок качества данных на основе музыкального датасета. Список ограничений — после описания шагов.
Сохраните полученный набор проверок в JSON-файл.
Сгенерируйте HTML-отчёт (Data Docs) по результатам проверки, чтобы убедиться, что проверка корректна.
Сохраните результаты проверки в JSON-файл.
Оформите результат. Все файлы, созданные при выполнении задания, поместите в папку с названием SurnameName_Task4.
Требования к ожидаемой схеме данных музыкального датасета

Пропуски в данных запрещены.
Порядок колонок не важен.
Обязательно должны быть все указанные колонки.
Лишние колонки не запрещены.
Ограничения на колонки представлены в таблице ниже.
Колонка Тип Ограничение
track_id str Длина строки строго 22 символа
artists str Длина строки от 2 до 512 символов включительно
album_name str Длина строки от 2 до 512 символов включительно
track_name str Длина строки от 2 до 512 символов включительно
popularity int Значения от 0 до 100 включительно
duration_ms int Значения от 0 не включительно до 5237760 включительно
explicit bool
danceability float Значения от 0 до 1 включительно
energy float Значения от 0 до 1 включительно
key int Значения от 0 до 11 включительно
loudness float Значения от -45 до 5 включительно
mode float Значения от 0 до 1 включительно
speechiness float Значения от 0 до 1 включительно
acousticness float Значения от 0 до 1 включительно
instrumentalness float Значения от 0 до 1 включительно
liveness float Значения от 0 до 1 включительно
valence float Значения от 0 до 1 включительно
tempo float Значения от 0 до 256 включительно;
time_signature int Значения от 0 до 5
track_genre str Ограничение на 114 уникальных жанров музыки
Следуй стилю максимально похожему в файле SharafutdinovNil_Task3.ipynb , можешь копировть блоки кода прямо оттуда , чтобы было максимально похоже, например проверка для поля track_genre .

In [8]:
import pandas as pd
import great_expectations as gx
import json
from datetime import datetime

In [9]:
df = pd.read_csv('dataset1.csv')
print(df.shape)

(114000, 21)


In [10]:
columns_to_drop = [col for col in df.columns if col.startswith('Unnamed') or col == 'index']
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
print(df.shape)

(114000, 20)


In [11]:
UNIQUE_GENRES = set(df['track_genre'].dropna().unique())
n_genres = len(UNIQUE_GENRES)
print(f'Уникальных жанров: {n_genres}')

Уникальных жанров: 114


In [12]:
# Создаём file context для сохранения всех артефактов
context = gx.get_context(mode='file', context_root_dir='great_expectations')
datasource = context.data_sources.add_pandas(name='music_data_source')
asset = datasource.add_dataframe_asset(name='music_data_asset')
batch_request = asset.build_batch_request(options={'dataframe': df})

DataContextError: Can not write the fluent datasource music_data_source because a datasource of that name already exists in the data context.

In [6]:
expectation_suite = gx.ExpectationSuite(name='music_data_expectations')
expectation_suite = context.suites.add(expectation_suite)

### Добавление ожиданий в Suite

In [7]:
# Проверка обязательных колонок
required_columns = [
    'track_id', 'artists', 'album_name', 'track_name', 'popularity',
    'duration_ms', 'explicit', 'danceability', 'energy', 'key',
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature', 'track_genre'
]

for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column=col)
    )

In [8]:
# Проверка на отсутствие пропусков
for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column=col)
    )

In [9]:
# Проверка track_id: длина строки строго 22 символа
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_id', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToEqual(column='track_id', value=22)
)

ExpectColumnValueLengthsToEqual(id='d98a60c7-6b0d-40f0-98e7-d751a0ef6485', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='track_id', mostly=1, row_condition=None, condition_parser=None, value=22.0)

In [10]:
# Проверка artists: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='artists', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='artists', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='9184aebb-ec49-4d9b-8c6e-dd2fe6ac5104', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='artists', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [11]:
# Проверка album_name: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='album_name', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='album_name', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='b4b5f143-7b74-4d74-b454-8cc41c54c5a9', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='album_name', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [12]:
# Проверка track_name: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_name', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='track_name', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='12c185a7-fa85-481c-8ead-a393b582ab2b', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='track_name', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [13]:
# Проверка popularity: int от 0 до 100
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='popularity', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='popularity', min_value=0, max_value=100, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='14eb795c-d456-4823-9f3a-51886b0d9e16', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='popularity', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=100.0, strict_min=False, strict_max=False)

In [14]:
# Проверка duration_ms: int от 0 (не включительно) до 5237760 (включительно)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='duration_ms', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='duration_ms', min_value=0, max_value=5237760, strict_min=True, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='ac81ff36-5dd0-4276-b686-e3f47b96cc90', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='duration_ms', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=5237760.0, strict_min=True, strict_max=False)

In [15]:
# Проверка explicit: bool
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='explicit', type_='bool')
)

ExpectColumnValuesToBeOfType(id='92a6be5f-2e59-4311-a7a8-6d1749e9cf1d', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='explicit', mostly=1, row_condition=None, condition_parser=None, type_='bool')

In [16]:
# Проверка danceability: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='danceability', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='danceability', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='f06367eb-ed97-47ca-8678-69beb335217e', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='danceability', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [17]:
# Проверка energy: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='energy', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='energy', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='8d1b8896-68c7-4913-8c97-5fcaa6b8ec74', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='energy', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [18]:
# Проверка key: int от 0 до 11
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='key', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='key', min_value=0, max_value=11, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='debfdaf2-ad8c-4f34-9f9e-1546bdb6d8e8', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='key', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=11.0, strict_min=False, strict_max=False)

In [19]:
# Проверка loudness: float от -45 до 5
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='loudness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='loudness', min_value=-45, max_value=5, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='61031cd2-7b59-4766-82c1-25e097d55433', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='loudness', mostly=1, row_condition=None, condition_parser=None, min_value=-45.0, max_value=5.0, strict_min=False, strict_max=False)

In [20]:
# Проверка mode: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='mode', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='mode', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='ff04f8b6-4d24-4b8f-bc02-5a7b039995b3', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='mode', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [21]:
# Проверка speechiness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='speechiness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='speechiness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='d5f471cb-993c-43be-97a7-e0b1b68e59a2', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='speechiness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [22]:
# Проверка acousticness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='acousticness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='acousticness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='36f83967-a922-4dd0-908e-4229dfc1515b', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='acousticness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [23]:
# Проверка instrumentalness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='instrumentalness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='instrumentalness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='7332fb69-5db0-4d57-8e52-5e8cbfac2f75', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='instrumentalness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [24]:
# Проверка liveness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='liveness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='liveness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='42000025-00df-404e-ae35-9af2ddc28a9a', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='liveness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [25]:
# Проверка valence: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='valence', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='valence', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='926d250a-e874-4caa-8bfc-83f823ff040e', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='valence', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [26]:
# Проверка tempo: float от 0 до 256
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='tempo', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='tempo', min_value=0, max_value=256, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='89cde884-fbc7-4562-858e-0f233ce8ae0a', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='tempo', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=256.0, strict_min=False, strict_max=False)

In [27]:
# Проверка time_signature: int от 0 до 5
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='time_signature', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='time_signature', min_value=0, max_value=5, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='36128c9c-883e-4309-8bf8-582c39e3908a', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='time_signature', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=5.0, strict_min=False, strict_max=False)

In [28]:
# Проверка track_genre: str, ограничение на 114 уникальных жанров
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_genre', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(column='track_genre', value_set=list(UNIQUE_GENRES))
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnUniqueValueCountToBeBetween(
        column='track_genre', min_value=n_genres, max_value=n_genres, strict_min=False, strict_max=False
    )
)
print(f'Количество уникальных жанров: {n_genres}')

Количество уникальных жанров: 114


### Сохранение Expectation Suite в JSON

In [29]:
# Сохраняем suite в context (уже сохранён при add_expectation)
# Сохраняем в JSON файл
suite_json = expectation_suite.to_json_dict()
with open('music_data_expectations.json', 'w', encoding='utf-8') as f:
    json.dump(suite_json, f, indent=2, ensure_ascii=False)
print('Expectation Suite сохранён в music_data_expectations.json')
print(f'Количество ожиданий: {len(expectation_suite.expectations)}')

Expectation Suite сохранён в music_data_expectations.json
Количество ожиданий: 80


### Запуск проверки и получение результатов

In [30]:
batch = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite.name
)

validation_result = batch.validate()
print(f'Проверка завершена: {validation_result.success}')

Calculating Metrics:   0%|          | 0/130 [00:00<?, ?it/s]

Проверка завершена: False


### Сохранение результатов проверки в JSON

In [31]:
results_dict = validation_result.to_json_dict()
with open('music_data_validation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_dict, f, indent=2, ensure_ascii=False)
print('Результаты проверки сохранены в music_data_validation_results.json')

Результаты проверки сохранены в music_data_validation_results.json


### Генерация HTML-отчёта (Data Docs)

In [32]:
context.build_data_docs()
print('HTML-отчёт сгенерирован в папке great_expectations/uncommitted/data_docs/')

HTML-отчёт сгенерирован в папке great_expectations/uncommitted/data_docs/

In [33]:
import os

data_docs_path = os.path.join(os.getcwd(), 'great_expectations', 'uncommitted', 'data_docs')
for root, dirs, files in os.walk(data_docs_path):
    for file in files:
        if file.endswith('.html'):
            print(f'HTML отчёт: {os.path.join(root, file)}')

HTML отчёт: c:\Projects\yp-sprint-5-practice-1\great_expectations\uncommitted\data_docs\local_site\index.html
HTML отчёт: c:\Projects\yp-sprint-5-practice-1\great_expectations\uncommitted\data_docs\local_site\expectations\music_data_expectations.html
